In [ ]:
!pip install scAnalysis

In [2]:
!git clone https://github.com/bowang-lab/scGPT.git

Cloning into 'scGPT'...
remote: Enumerating objects: 1128, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 1128 (delta 19), reused 33 (delta 15), pack-reused 1065 (from 1)
Receiving objects: 100% (1128/1128), 34.90 MiB | 23.68 MiB/s, done.
Resolving deltas: 100% (634/634), done.


In [ ]:
!pip install -e ./scGPT

In [4]:
import os
import sys
import json
import torch
import tarfile
import urllib.request
import numpy as np
import pandas as pd

In [5]:
sys.path.append("/content/scGPT")
import scgpt as scg

/content/scGPT/scgpt/model/flash_attn_compat.py:463: SyntaxWarning: invalid escape sequence '\.'
  >>> # rules = {r"self_attn\.Wqkv\.": "self_attn._impl.Wqkv.",


In [6]:
from scAnalysis import (
    sc_io,
    preprocessing,
    quality_control,
    cell_cycle,
    batch_correction,
    dimensionality,
    clustering,
    trajectory,
    differential,
    enrichment,
    visualization,
    interactive_viz,
    imputation,
)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [8]:
model_dir = "/content/drive/MyDrive/scGPT_human"

In [9]:
def get_pbmc3k_data():
    url = "https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"
    data_dir = "./data"
    filepath = os.path.join(data_dir, "pbmc3k.tar.gz")
    extract_path = os.path.join(data_dir, "pbmc3k_extracted")

    if not os.path.exists(data_dir):
        os.makedirs(data_dir)

    if not os.path.exists(extract_path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response, open(filepath, "wb") as out_file:
            out_file.write(response.read())

        with tarfile.open(filepath, "r:gz") as tar:
            tar.extractall(path=extract_path)

    return os.path.join(extract_path, "filtered_gene_bc_matrices", "hg19")

data_path = get_pbmc3k_data()
data = sc_io.read_10x_mtx(data_path)
data.var.index = sc_io._make_unique(data.var.index.values)

/tmp/ipykernel_392/4141867233.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


IO: data from './data/pbmc3k_extracted/filtered_gene_bc_matrices/hg19' …
IO: Loaded 2,700 cells × 32,738 genes.


In [10]:
preprocessing.calculate_qc_metrics(data, qc_vars=["MT-"])

data = preprocessing.filter_cells(data, min_genes=200)
data = preprocessing.filter_genes(data, min_cells=3)

preprocessing.normalize_total(data, target_sum=1e4)
preprocessing.log1p(data)
imputation.impute_wnid(data, k=3, dropout_thresh=0.9, n_pcs=30)

preprocessing.highly_variable_genes(data, n_top_genes=2000)

data.var["gene_name"] = data.var.index.tolist()

filter_cells: keeping 2,700 / 2,700 cells.
filter_genes: keeping 13,714 / 32,738 genes.
HVG: identified 2,000 highly variable genes.


In [11]:
vocab_file = os.path.join(model_dir, "vocab.json")
vocab = scg.tokenizer.GeneVocab.from_file(vocab_file)

In [12]:
gene_ids = np.array(vocab(data.var["gene_name"].tolist()))

In [13]:
args_file = os.path.join(model_dir, "args.json")
with open(args_file, "r") as f:
    model_args = json.load(f)

In [14]:
model_path = os.path.join(model_dir, "best_model.pt")

In [15]:
try:
    from scgpt.model import TransformerModel
except ImportError:
    from scgpt.models import TransformerModel

In [16]:
model = TransformerModel(
    ntoken=len(vocab),
    d_model=model_args.get("embsize", 512),
    nhead=model_args.get("nheads", 8),
    d_hid=model_args.get("d_hid", 512),
    nlayers=model_args.get("nlayers", 12),
    vocab=vocab,
    pad_value=model_args.get("pad_value", vocab["<pad>"] if "<pad>" in vocab else 0)
)

In [17]:
state_dict = torch.load(model_path, map_location=device)
harmonized_state_dict = {}

In [18]:
for key, value in state_dict.items():
    if "Wqkv.weight" in key:
        new_key = key.replace("Wqkv.weight", "in_proj_weight")
        harmonized_state_dict[new_key] = value
    elif "Wqkv.bias" in key:
        new_key = key.replace("Wqkv.bias", "in_proj_bias")
        harmonized_state_dict[new_key] = value
    else:
        harmonized_state_dict[key] = value

missing_keys, unexpected_keys = model.load_state_dict(harmonized_state_dict, strict=False)

In [19]:
model.to(device)
model.eval()

TransformerModel(
  (encoder): GeneEncoder(
    (embedding): Embedding(60697, 512, padding_idx=60694)
    (enc_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.5, inplace=False)
    (linear1): Linear(in_features=1, out_features=512, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.5, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, el

In [20]:
with torch.no_grad():
    try:
        from scgpt.tasks.cell_emb import get_batch_cell_embeddings
        cell_embeddings = get_batch_cell_embeddings(
            adata=data,
            cell_embedding_mode="cls",
            model=model,
            vocab=vocab,
            max_length=1200,
            batch_size=64,
            model_configs=model_args,
            gene_ids=gene_ids,
            use_batch_labels=False
        )
    except Exception as e:
        print(f"ERROR: Failed during embedding extraction. Details: {e}")
        raise

/content/scGPT/scgpt/tasks/cell_emb.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
Embedding cells: 100%|██████████| 43/43 [00:26<00:00,  1.63it/s]


In [21]:
data.obsm["X_scGPT"] = cell_embeddings
data.obsm["X_pca"] = cell_embeddings

In [22]:
dimensionality.neighbors(data, n_neighbors=15)

Neighbors: k=15, metric='euclidean' …
Neighbors: graph built (2,700 cells).


SingleCellDataset object with n_obs × n_vars = 2700 × 13714
    obs: barcode, n_genes_by_counts, total_counts, pct_counts_MT-
    var: gene_ids, gene_symbols, n_cells, means, dispersions, dispersions_norm, highly_variable, gene_name
    uns: neighbors
    obsm: X_scGPT, X_pca
    Memory (X): 32.08 MB

In [23]:
dimensionality.run_umap(data, min_dist=0.3)

UMAP: min_dist=0.3, n_components=2 …


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


SingleCellDataset object with n_obs × n_vars = 2700 × 13714
    obs: barcode, n_genes_by_counts, total_counts, pct_counts_MT-
    var: gene_ids, gene_symbols, n_cells, means, dispersions, dispersions_norm, highly_variable, gene_name
    uns: neighbors
    obsm: X_scGPT, X_pca, X_umap
    Memory (X): 32.08 MB

In [24]:
clustering.cluster_leiden(data, resolution=0.5, key_added="leiden")

Clustering: Leiden resolution=0.5 …
Leiden: found 9 clusters.


SingleCellDataset object with n_obs × n_vars = 2700 × 13714
    obs: barcode, n_genes_by_counts, total_counts, pct_counts_MT-, leiden
    var: gene_ids, gene_symbols, n_cells, means, dispersions, dispersions_norm, highly_variable, gene_name
    uns: neighbors
    obsm: X_scGPT, X_pca, X_umap
    Memory (X): 32.08 MB

In [25]:
output_file = "/content/drive/MyDrive/pbmc3k_scgpt_scanalyzer_processed.h5ad"
sc_io.write_h5ad(data, output_file)

IO: Writing H5AD -> '/content/drive/MyDrive/pbmc3k_scgpt_scanalyzer_processed.h5ad' ...
IO: Wrote 2,700 cells × 13,714 genes.


In [26]:
visualization.plot_umap(
    data,
    color="leiden",
    title="scGPT Zero-shot Embeddings via scAnalyzer",
    save="/content/scGPT_umap_scanalyzer.png"
)

Saved → /content/scGPT_umap_scanalyzer.png


<Axes: title={'center': 'scGPT Zero-shot Embeddings via scAnalyzer'}, xlabel='X_umap 1', ylabel='X_umap 2'>